In [55]:
# Step 1: Install scikit-learn
!pip install -U scikit-learn

In [56]:
# Step 2: Import libraries
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
# Step 3: Load and clean dataset
df = pd.read_csv('/content/tags.csv')
df = df.dropna(subset=['tag'])

In [58]:
# Step 4: Use a sample of top-tagged movies to reduce memory usage
tag_counts = df['movieId'].value_counts().head(1000).index
df = df[df['movieId'].isin(tag_counts)]

In [59]:
movie_tags = df.groupby('movieId')['tag'].apply(lambda x: ', '.join(x)).reset_index()


In [60]:
# Step 6: TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(movie_tags['tag'])

In [61]:
# Step 7: Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [62]:
from tabulate import tabulate  # Make sure this is installed

def recommend_movies(movie_index, top_n=10):
    similar_scores = list(enumerate(cosine_sim[movie_index]))
    sorted_scores = sorted(similar_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    movie_id = movie_tags.iloc[movie_index]['movieId']
    print(f"\nMovies similar to Movie ID: {movie_id}\n")

    table_data = []
    for i, score in sorted_scores:
        table_data.append([
            movie_tags.iloc[i]['movieId'],
            round(score, 3),
            movie_tags.iloc[i]['tag'][:100] + '...' if len(movie_tags.iloc[i]['tag']) > 100 else movie_tags.iloc[i]['tag']
        ])

    headers = ['Movie ID', 'Similarity', 'Tags']
    print(tabulate(table_data, headers=headers, tablefmt='fancy_grid'))


In [63]:
# Step 9: Run the model
recommend_movies(0)


Movies similar to Movie ID: 1

╒════════════╤══════════════╤═════════════════════════════════════════════════════════════════════════════════════════════════════════╕
│   Movie ID │   Similarity │ Tags                                                                                                    │
╞════════════╪══════════════╪═════════════════════════════════════════════════════════════════════════════════════════════════════════╡
│       3114 │        0.938 │ Disney, Owned, imdb top 250, original, animation, Disney, Pixar, bright, DARING RESCUES, fanciful, h... │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       4886 │        0.813 │ Owned, imdb top 250, Katottava, funny, cute, hilarious, innovative, animation, Disney, Pixar, Animat... │
├────────────┼──────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│      78499 │  